# 03 · Neural networks: direct vs residual learning

Owner: ML lead.

Same features and architecture as Qiao & Wan (2024). Early stopping on 2019; 2020 untouched until the final evaluation.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from config import *
from src import load, pairs as pairs_mod, models, evaluate

In [ ]:
pairs = pd.read_parquet(PROCESSED_DIR / "pairs_with_baselines.parquet")
train, val, test = (pairs[pairs.split == s] for s in ("train", "val", "test"))

In [ ]:
nn_resid  = models.NeuralHedger(residual=True).fit(train, val=val)
nn_direct = models.NeuralHedger(residual=False).fit(train, val=val)

In [ ]:
for m in (nn_resid, nn_direct):
    h = pd.DataFrame(m.history_, columns=["epoch", "train", "val"]).set_index("epoch")
    h.plot(title=f"{m.name}: weighted MSE"); plt.show()

In [ ]:
pairs = evaluate.add_predictions(pairs, {"nn_resid": nn_resid, "nn_direct": nn_direct})
MODELS = ["bs", "hw", "gbm", "nn_direct", "nn_resid"]
tables = evaluate.standard_report(pairs, MODELS, TABLES_DIR / "neural")
tables["by_split_type"]

In [ ]:
# crisis window
evaluate.gain_table(evaluate.crisis_window(pairs), MODELS, ["type"])

In [ ]:
pairs.to_parquet(PROCESSED_DIR / "pairs_with_all_models.parquet", index=False)
import torch; torch.save(nn_resid.net_.state_dict(), ROOT / "nn_resid.pt")